# 03 - Diffusion Post-Process for Reconstruction Renders

This notebook applies diffusion models to improve the look of rendered outputs.
It is meant for visualization, not for geometry accuracy.

Two options are included:
- Inpainting to fill missing texture or clean artifacts
- Depth-to-image (structure-aware enhancement)


In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw

ROOT = Path.cwd()
RENDER_INPUT = ROOT / "outputs" / "ETs_cluster_0001" / "renders"
OUT_DIR = ROOT / "outputs" / "ETs_cluster_0001" / "diffusion"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("RENDER_INPUT:", RENDER_INPUT)
print("OUT_DIR:", OUT_DIR)

images = sorted(RENDER_INPUT.glob("*.png"))
print("num input images:", len(images))
if not images:
    print("Place renders in RENDER_INPUT and re-run this notebook.")


## Option A: Inpainting

This uses a text prompt + mask to clean or fill artifacts.


In [ ]:
# Uncomment to install (requires network)
# !pip install diffusers transformers accelerate safetensors

from diffusers import StableDiffusionInpaintPipeline
import torch

MODEL_ID = "runwayml/stable-diffusion-inpainting"
PIPE = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
PIPE = PIPE.to("cuda" if torch.cuda.is_available() else "cpu")

PROMPT = "clean photorealistic texture, remove artifacts"
NEG_PROMPT = "low quality, blurry, deformed"


In [ ]:
def load_image(path: Path):
    return Image.open(path).convert("RGB")

def dummy_mask(image: Image.Image):
    # Replace this with your own mask. White pixels are inpainted.
    w, h = image.size
    mask = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(mask)
    box = (int(w * 0.25), int(h * 0.25), int(w * 0.75), int(h * 0.75))
    draw.rectangle(box, fill=255)
    return mask

# Example single-image run
# img_path = images[0]
# image = load_image(img_path)
# mask = dummy_mask(image)
# result = PIPE(prompt=PROMPT, negative_prompt=NEG_PROMPT, image=image, mask_image=mask).images[0]
# result.save(OUT_DIR / f"{img_path.stem}_inpaint.png")


## Option B: Depth-to-Image (structure-aware enhancement)

This keeps geometry cues while improving appearance.


In [ ]:
from diffusers import StableDiffusionDepth2ImgPipeline
import torch

DEPTH_MODEL = "stabilityai/stable-diffusion-2-depth"
DEPTH_PIPE = StableDiffusionDepth2ImgPipeline.from_pretrained(
    DEPTH_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
DEPTH_PIPE = DEPTH_PIPE.to("cuda" if torch.cuda.is_available() else "cpu")

PROMPT = "high quality photorealistic render of the scene"

# Example single-image run
# img_path = images[0]
# image = load_image(img_path)
# result = DEPTH_PIPE(prompt=PROMPT, image=image).images[0]
# result.save(OUT_DIR / f"{img_path.stem}_depth2img.png")


## Batch processing helper


In [ ]:
def batch_inpaint():
    for img_path in images:
        image = load_image(img_path)
        mask = dummy_mask(image)
        result = PIPE(prompt=PROMPT, negative_prompt=NEG_PROMPT, image=image, mask_image=mask).images[0]
        result.save(OUT_DIR / f"{img_path.stem}_inpaint.png")

def batch_depth2img():
    for img_path in images:
        image = load_image(img_path)
        result = DEPTH_PIPE(prompt=PROMPT, image=image).images[0]
        result.save(OUT_DIR / f"{img_path.stem}_depth2img.png")
